# Doing a 1 basin LSTM


In [16]:
import xarray as xr

ds = xr.open_dataset("Caravan-nc/timeseries/netcdf/camels/camels_01013500.nc")

def add_water_year(ds):
    '''
    Water year is defined as Oct 1 to Sep 30.
    Oct, Nov, Dec are labelled as regular year + 1
    '''
    t = ds['date']
    water_year = t.dt.year + (t.dt.month >= 10)
    new_ds = ds.assign_coords(water_year=water_year)
    return new_ds

ds = add_water_year(ds)

# Get rid of NaN values
ds = ds.dropna(dim='date', 
               subset=['streamflow', 
                       'total_precipitation_sum',
                       'temperature_2m_mean',
                       'surface_net_solar_radiation_mean',
                       'surface_net_thermal_radiation_mean',
                       'dewpoint_temperature_2m_mean',
                       'u_component_of_wind_10m_mean',
                       'v_component_of_wind_10m_mean'], # From which features
                 how='any') # Any of the subsets


- LSTM definition.
- Train on all days; predict on all days; group test/train days into water years and calculate metrics from there.

In [ ]:
import torch
import torch.nn as nn

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size, dropout=0.0):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size, # the dimensionality of ct and ht
            num_layers=num_layers,
            batch_first=True,
            bidirectional=False,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.output = nn.Linear(hidden_size, output_size) # This is the fully connected output layer turning the h_t into an output

    def forward(self, x, state=None): # This is what happens when you call model(x); state=None means if there is no supplied LSTM state then start from scratch
        '''
        x is in Batch, Time, input size dimensions
        '''
        out, (h_n, c_n) = self.lstm(x, state)       # out is in Batch, Time, hidden_size dimensions
        y = self.output(out[:, -1, :])                  # last time step yields B, hidden which transforms into Batch, output size
        return y, (h_n, c_n)


# PyTorch dataloader needs data to be able to do certain things
from torch.utils.data import Dataset, DataLoader

class BasinDataset(Dataset):
    def __init__(self, features, streamflow, water_year, seq_len=365):
        '''
        features in time, num_features dimensional numpy array
        streamflow in time dimensional numpy array
        '''
        self.x = torch.tensor(features, dtype=torch.float32)
        self.y = torch.tensor(streamflow, dtype=torch.float32)
        self.seq_len = seq_len

    def __len__(self): # this is one of those things dataloader needs from a dataset
        return len(self.y) - self.seq_len # There are total - 365 sequences available

    def __getitem__(self, i):
        x_seq = self.x[i : i + self.seq_len]
        y_val = self.y[i + self.seq_len]
        return x_seq, y_val

# Define the normalised square loss for the training
def nse_loss(y_pred, y_obs, std_train):
    return torch.mean((y_pred - y_obs) ** 2) / (std_train + 1e-6) ** 2

- Now do the datasplit


In [ ]:
import numpy as np

last_training_year = 1999 # 20 after 1980 inclusive
all_water_years = np.array(ds['water_year'])
split_idx = np.searchsorted(all_water_years, last_training_year + 1e-4)
# split idx is the first datapoint of the test set

# # Initially just an easy split by length
# dataset_len = len(ds['streamflow'].values)
# trainsplit_idx = int(dataset_len//2)

# basin01013500_features = np.stack([
#     ds['total_precipitation_sum'].values,
#     ds['temperature_2m_mean'].values,
#     ds['surface_net_solar_radiation_mean'].values,
#     ds['surface_net_thermal_radiation_mean'].values,
#     ds['dewpoint_temperature_2m_mean'].values,
#     np.sqrt(ds['u_component_of_wind_10m_mean']**2 + ds['v_component_of_wind_10m_mean']**2).values
#     ], axis=1)




np.int64(1999)